In [7]:
from langgraph.graph import StateGraph,START,END
import os
from dotenv import load_dotenv
from typing import TypedDict,Literal
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel,Field

In [8]:
# Initialize Gemini
load_dotenv()
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

In [15]:
class SentimentSchema(BaseModel):
    sentiment: Literal["positive","negative"] = Field(description='Sentiment of the review')

class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')
    


In [16]:
structured_model = llm.with_structured_output(SentimentSchema)
structured_model2 = llm.with_structured_output(DiagnosisSchema)


In [ ]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal["positive","negative"]
    diagnosis: dict
    response : str

In [ ]:
def find_sentiment(state:ReviewState) -> ReviewState:
    prompt = f"For the following review find out the sentiment \n {state['review']}"
    sentiment =structured_model.invoke(prompt).sentiment 
    return {'sentiment':sentiment}

def postive_response(state:ReviewState)-> ReviewState:
    prompt = f"""Write a warm thank-you message in response to this review:\n\n\"{state['review']}\"\n Also, kindly ask the user to leave feedback on our website."""
    positive_res = llm.invoke(prompt).content
    return {'response':positive_res}
    
def run_diagnosis(state: ReviewState):

    prompt = f"""Diagnose this negative review:\n\n{state['review']}\n""Return issue_type, tone, and urgency."""
    response = structured_model2.invoke(prompt)
    return {'diagnosis':response.model_dump()}

def negative_response(state:ReviewState) -> ReviewState:
    diagnosis = state['diagnosis']

    prompt = f"""You are a support assistant.
    The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
    Write an empathetic, helpful resolution message."""

    response = llm.invoke(prompt).content
    return {'response':response}
   
def router(state:ReviewState)->Literal['postive_response','run_diagnosis']:

    if state['sentiment'] == 'positive':
        return 'postive_response'
    else:
        return 'run_diagnosis'
    


In [ ]:
graph  = StateGraph(ReviewState)

graph.add_node('find_sentiment',find_sentiment)



graph.add_edge(START,'find_sentiment')
graph.add_edge

SentimentSchema(sentiment='positive')

SentimentSchema(sentiment='positive')